# BERTScore F1 Calculator
Upload `dataset_model_final.csv` and this will compute BERTScore F1 for all 4 output columns.

**Runtime: CPU is fine. No GPU needed.** Takes ~5-10 minutes.

In [ ]:
# Cell 1: Install
!pip install -q bert_score pandas
print('Done!')

In [ ]:
# Cell 2: Upload your CSV
from google.colab import files
uploaded = files.upload()  # Click 'Choose Files' and select dataset_model_final.csv

In [ ]:
# Cell 3: Load data
import pandas as pd

df = pd.read_csv('dataset_model_final.csv')
print(f'Loaded {len(df)} rows')
print(f'Columns: {list(df.columns)}')

# The 4 output columns to evaluate
output_cols = ['output_generic', 'output_financial', 'output_financial_t5xl', 'output_llm']

# Check they exist
for col in output_cols:
    if col in df.columns:
        print(f'  {col}: {df[col].notna().sum()} non-empty rows')
    else:
        print(f'  {col}: NOT FOUND - will skip')

# Reference column
ref_col = 'text_simplified_reference'
print(f'  {ref_col}: {df[ref_col].notna().sum()} non-empty rows')

In [ ]:
# Cell 4: Compute BERTScore F1 for all output columns
from bert_score import score
import numpy as np

results = {}

for col in output_cols:
    if col not in df.columns:
        print(f'Skipping {col} (not found)')
        continue

    print(f'\nComputing BERTScore for {col}...')

    # Get outputs and references as lists of strings
    # Fill any NaN with empty string to avoid errors
    outputs = df[col].fillna('').astype(str).tolist()
    references = df[ref_col].fillna('').astype(str).tolist()

    # Compute BERTScore (comparing output against reference)
    P, R, F1 = score(
        outputs,
        references,
        lang='en',
        verbose=True,
        batch_size=32
    )

    f1_scores = F1.numpy()
    mean_f1 = np.mean(f1_scores)
    std_f1 = np.std(f1_scores)

    results[col] = {
        'mean': round(mean_f1, 4),
        'std': round(std_f1, 4),
        'min': round(np.min(f1_scores), 4),
        'max': round(np.max(f1_scores), 4),
        'scores': f1_scores
    }

    # Also save per-row scores to the dataframe
    df[f'bertscore_f1_{col}'] = f1_scores

    print(f'  {col}: BERTScore F1 = {mean_f1:.4f} +/- {std_f1:.4f}')

print('\nDone!')

In [ ]:
# Cell 5: Summary table
print('=' * 60)
print('BERTScore F1 Summary')
print('=' * 60)
print(f'{"Model":<30} {"Mean":>8} {"SD":>8} {"Min":>8} {"Max":>8}')
print('-' * 60)
for col, r in results.items():
    print(f'{col:<30} {r["mean"]:>8.4f} {r["std"]:>8.4f} {r["min"]:>8.4f} {r["max"]:>8.4f}')
print('=' * 60)
print()
print('Copy-paste for your report:')
print()
for col, r in results.items():
    print(f'  {col}: {r["mean"]:.4f} +/- {r["std"]:.4f}')

In [ ]:
# Cell 6: Save results

# Save summary table
summary_rows = []
for col, r in results.items():
    summary_rows.append({
        'model': col,
        'bertscore_f1_mean': r['mean'],
        'bertscore_f1_sd': r['std'],
        'bertscore_f1_min': r['min'],
        'bertscore_f1_max': r['max']
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('bertscore_summary.csv', index=False)
print('Saved bertscore_summary.csv')

# Save full dataset with per-row BERTScore columns added
df.to_csv('dataset_model_final_with_bertscore.csv', index=False)
print('Saved dataset_model_final_with_bertscore.csv')

# Download both files
files.download('bertscore_summary.csv')
files.download('dataset_model_final_with_bertscore.csv')
print('\nDownload started!')